# Interactive Inference Demo

This notebook provides an interactive interface to test the trained coding agent.

**Features:**
- Generate code from natural language descriptions
- Test different languages (.NET, Angular, SQL)
- Adjust generation parameters
- Compare multiple generations

In [1]:
import torch
from transformers import T5ForConditionalGeneration, RobertaTokenizer
from pathlib import Path
import json
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set paths
MODEL_DIR = Path('../models').resolve()

print("Interactive demo environment ready!")

Using device: cpu
Interactive demo environment ready!


## 1. Load Trained Model

In [2]:
# Load model and tokenizer
model_path = (MODEL_DIR / 'best_model').resolve()
# Check if model exists first to avoid HFValidationError on Windows
if not model_path.exists():
    raise FileNotFoundError(f"Model not found at {model_path}. Please train the model first.")
model = T5ForConditionalGeneration.from_pretrained(model_path.as_posix())
tokenizer = RobertaTokenizer.from_pretrained(model_path.as_posix())
model = model.to(device)
model.eval()

print(f"✓ Model loaded from: {model_path}")
print(f"✓ Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Load config
with open(MODEL_DIR / 'training_config.json', 'r') as f:
    config = json.load(f)

print("✓ Configuration loaded")

HFValidationError: Repo id must use alphanumeric chars or '-', '_', '.', '--' and '..' are forbidden, '-' and '.' cannot start or end the name, max length is 96: 'D:\Mugil-Agent\simple-coding-agent\models\best_model'.

## 2. Define Generation Function

In [ ]:
def generate_code(language, framework, task, context="", 
                  max_length=512, num_beams=4, temperature=0.7, top_p=0.95):
    """
    Generate code based on description
    
    Args:
        language: Programming language (csharp, typescript, sql)
        framework: Framework (dotnet8, angular, mssql)
        task: Description of what to generate
        context: Additional context (optional)
        max_length: Maximum output length
        num_beams: Number of beams for beam search
        temperature: Sampling temperature
        top_p: Top-p sampling parameter
    
    Returns:
        Generated code as string
    """
    # Create input prompt
    prompt = f"Language: {language}\n"
    prompt += f"Framework: {framework}\n"
    prompt += f"Task: {task}\n"
    if context:
        prompt += f"Context: {context}\n"
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        max_length=config['max_input_length'],
        truncation=True
    ).to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_length=max_length,
            num_beams=num_beams,
            temperature=temperature,
            top_p=top_p,
            early_stopping=True,
            do_sample=temperature > 0
        )
    
    # Decode
    generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return generated_code

print("✓ Generation function defined")

## 3. Example Generations

### Example 1: .NET Core 8+ Controller

In [ ]:
code = generate_code(
    language="csharp",
    framework="dotnet8",
    task="Create a REST API controller for managing products with CRUD operations",
    context="Use Entity Framework Core and async methods"
)

print("Generated C# Controller:")
print("=" * 80)
print(code)
print("=" * 80)

### Example 2: Angular Service

In [ ]:
code = generate_code(
    language="typescript",
    framework="angular",
    task="Create a service to handle user authentication with login and logout methods",
    context="Use HttpClient and store JWT token"
)

print("Generated Angular Service:")
print("=" * 80)
print(code)
print("=" * 80)

### Example 3: SQL Query

In [ ]:
code = generate_code(
    language="sql",
    framework="mssql",
    task="Create a stored procedure to get sales report by date range",
    context="Include total sales, number of orders, and average order value"
)

print("Generated SQL Stored Procedure:")
print("=" * 80)
print(code)
print("=" * 80)

## 4. Interactive Widget Interface

In [ ]:
# Create interactive widgets
language_dropdown = widgets.Dropdown(
    options=['csharp', 'typescript', 'sql'],
    value='csharp',
    description='Language:',
    style={'description_width': 'initial'}
)

framework_dropdown = widgets.Dropdown(
    options=['dotnet8', 'angular', 'mssql'],
    value='dotnet8',
    description='Framework:',
    style={'description_width': 'initial'}
)

task_text = widgets.Textarea(
    value='Create a simple API controller',
    placeholder='Describe what code you want to generate...',
    description='Task:',
    layout=widgets.Layout(width='600px', height='80px'),
    style={'description_width': 'initial'}
)

context_text = widgets.Textarea(
    value='',
    placeholder='Additional context (optional)...',
    description='Context:',
    layout=widgets.Layout(width='600px', height='60px'),
    style={'description_width': 'initial'}
)

max_length_slider = widgets.IntSlider(
    value=512,
    min=128,
    max=1024,
    step=64,
    description='Max Length:',
    style={'description_width': 'initial'}
)

num_beams_slider = widgets.IntSlider(
    value=4,
    min=1,
    max=8,
    step=1,
    description='Num Beams:',
    style={'description_width': 'initial'}
)

temperature_slider = widgets.FloatSlider(
    value=0.7,
    min=0.1,
    max=1.5,
    step=0.1,
    description='Temperature:',
    style={'description_width': 'initial'}
)

generate_button = widgets.Button(
    description='Generate Code',
    button_style='success',
    icon='code'
)

output_area = widgets.Output()

def on_generate_click(b):
    with output_area:
        output_area.clear_output()
        print("Generating code...\n")
        
        try:
            code = generate_code(
                language=language_dropdown.value,
                framework=framework_dropdown.value,
                task=task_text.value,
                context=context_text.value,
                max_length=max_length_slider.value,
                num_beams=num_beams_slider.value,
                temperature=temperature_slider.value
            )
            
            print("✓ Generation complete!\n")
            print("=" * 80)
            print("GENERATED CODE:")
            print("=" * 80)
            print(code)
            print("=" * 80)
        except Exception as e:
            print(f"✗ Error: {str(e)}")

generate_button.on_click(on_generate_click)

# Display interface
display(HTML("<h3>🤖 Coding Agent - Interactive Code Generator</h3>"))
display(widgets.VBox([
    widgets.HBox([language_dropdown, framework_dropdown]),
    task_text,
    context_text,
    widgets.HBox([max_length_slider, num_beams_slider, temperature_slider]),
    generate_button,
    output_area
]))

## 5. Batch Generation Test

In [ ]:
# Test multiple scenarios
test_scenarios = [
    {
        'language': 'csharp',
        'framework': 'dotnet8',
        'task': 'Create a middleware for logging HTTP requests',
        'context': 'Log request method, path, and response time'
    },
    {
        'language': 'typescript',
        'framework': 'angular',
        'task': 'Create a custom pipe to format currency',
        'context': 'Support multiple currency symbols'
    },
    {
        'language': 'sql',
        'framework': 'mssql',
        'task': 'Create a view to show top 10 customers by total purchases',
        'context': 'Include customer name, total orders, and total amount'
    }
]

print("Running batch generation test...\n")

for i, scenario in enumerate(test_scenarios, 1):
    print(f"\n{'='*80}")
    print(f"Test {i}: {scenario['language'].upper()} - {scenario['task']}")
    print(f"{'='*80}")
    
    code = generate_code(**scenario)
    
    print(f"\nGenerated Code:")
    print(code)
    print(f"\nCode length: {len(code)} characters")

print(f"\n{'='*80}")
print("Batch generation complete!")
print(f"{'='*80}")

## 6. Custom Prompt Testing

In [ ]:
# Try your own prompts here!
# Modify the parameters below and run the cell

my_language = "csharp"  # Options: csharp, typescript, sql
my_framework = "dotnet8"  # Options: dotnet8, angular, mssql
my_task = "Create a repository pattern implementation for database access"
my_context = "Use generic repository with async methods"

print("Generating custom code...\n")
custom_code = generate_code(
    language=my_language,
    framework=my_framework,
    task=my_task,
    context=my_context
)

print("=" * 80)
print("YOUR CUSTOM GENERATED CODE:")
print("=" * 80)
print(custom_code)
print("=" * 80)

## Summary

This interactive demo allows you to:
- ✓ Generate code for .NET Core 8+, Angular, and SQL
- ✓ Customize generation parameters
- ✓ Test different scenarios
- ✓ Use the interactive widget interface

### Next Steps:
1. Integrate the model into your development workflow
2. Use the Flask API (see `src/api.py`) for production deployment
3. Fine-tune further with your own codebase
4. Expand to support more languages and frameworks